In [3]:
import polars as pl
import numpy as np

## item_features.parquet

| Колонка | Тип | Описание |
|---------|-----|----------|
| `item_id` | UInt32 | Анонимизированный id объявления — ключ, на котором джойнятся кликстрим и таргет. |
| `vertical_id` | UInt32 | Анонимизированная вертикаль каталога (8 уникальных значений 0–7). Конкретные имена вертикалей не раскрываются. |
| `category_ext_y` | Int | Токенизированная категория (низкий уровень таксономии). |
| `region_id_y` | Int | Токенизированный регион (субъект РФ). |
| `loc_id_y` | Int | Токенизированный город / локация внутри региона. |
| `sid_0_y` | Int | Семантический id объявления, кодбук № 0. |
| `sid_1_y` | Int | Семантический id, кодбук № 1. |
| `sid_2_y` | Int | Семантический id, кодбук № 2. |
| `sid_3_y` | Int | Семантический id, кодбук № 3. |

`sid_0..sid_3` — это выход **Residual Quantization**-кодировщика над BERT-эмбеддингом текста объявления (title + description). Четыре последовательных кодбука дают «грубое → тонкое» представление контента. Семантически близкие объявления имеют близкие префиксы `sid`, поэтому эти колонки особенно полезны для кандидат-генерации в холодном старте.

In [4]:
items = pl.read_parquet('../data/item_features.parquet')

items = items.with_columns([
    pl.col('vertical_id').cast(pl.UInt8),
    pl.col('category_ext_y').cast(pl.UInt16),
    pl.col('region_id_y').cast(pl.UInt16),
    pl.col('loc_id_y').cast(pl.UInt16),
    pl.col('sid_0_y').cast(pl.UInt16),
    pl.col('sid_1_y').cast(pl.UInt16),
    pl.col('sid_2_y').cast(pl.UInt16),
    pl.col('sid_3_y').cast(pl.UInt16),
])

items

item_id,vertical_id,category_ext_y,region_id_y,loc_id_y,sid_0_y,sid_1_y,sid_2_y,sid_3_y
u32,u8,u16,u16,u16,u16,u16,u16,u16
140,0,32,34,1412,493,365,1001,724
142,0,31,38,1638,588,64,295,800
154,5,0,62,2775,584,460,313,365
182,0,14,34,1412,55,410,1022,500
205,0,11,63,2793,443,706,983,152
…,…,…,…,…,…,…,…,…
178328602,0,6,45,1982,801,82,938,963
178328646,0,30,34,1412,107,526,431,419
178328682,0,31,66,3003,170,712,136,598


In [5]:
items.describe()

statistic,item_id,vertical_id,category_ext_y,region_id_y,loc_id_y,sid_0_y,sid_1_y,sid_2_y,sid_3_y
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.78327659e8,1.78327659e8,1.78327659e8,1.78327659e8,1.78327659e8,1.78327659e8,1.78327659e8,1.78327659e8,1.78327659e8
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",8.9164e7,0.536398,16.0957,42.864792,1884.877108,490.032814,507.96893,514.314739,510.466889
"""std""",5.1479e7,1.425515,14.413107,20.251647,1019.645411,280.103723,300.050663,291.496354,297.101365
"""min""",0.0,0.0,0.0,2.0,8.0,0.0,0.0,0.0,0.0
"""25%""",4.4582204e7,0.0,5.0,34.0,1412.0,267.0,244.0,263.0,249.0
"""50%""",8.9164402e7,0.0,11.0,35.0,1562.0,437.0,505.0,511.0,511.0
"""75%""",1.33746614e8,0.0,22.0,63.0,2793.0,720.0,771.0,765.0,770.0
"""max""",1.7832885e8,7.0,51.0,86.0,9611.0,1023.0,1023.0,1023.0,1023.0


In [7]:
print(f"Уникальных токенезированных категорий - {items['category_ext_y'].n_unique()}")
print(f"Уникальных id регион - {items['region_id_y'].n_unique()}")
print(f"Уникальных id городов - {items['loc_id_y'].n_unique()}")
print(f"Уникальных токенезировнныхо писаний №0 - {items['sid_0_y'].n_unique()}")
print(f"Уникальных токенезировнныхо писаний №1 - {items['sid_1_y'].n_unique()}")
print(f"Уникальных токенезировнныхо писаний №2 - {items['sid_2_y'].n_unique()}")
print(f"Уникальных токенезировнныхо писаний №3 - {items['sid_3_y'].n_unique()}")

Уникальных токенезированных категорий - 52
Уникальных id регион - 85
Уникальных id городов - 6665
Уникальных токенезировнныхо писаний №0 - 1024
Уникальных токенезировнныхо писаний №1 - 1024
Уникальных токенезировнныхо писаний №2 - 1024
Уникальных токенезировнныхо писаний №3 - 1024


In [8]:
# КОличество значений по категориям товаров
items['vertical_id'].value_counts().sort('vertical_id')

vertical_id,count
u8,u32
0,153162868
1,8609
2,4947048
3,6197510
4,7540650
5,4143039
6,14042
7,2313893


## История пользователей для оторых нуно предсказать

| Колонка | Тип | Описание |
|---------|-----|----------|
| `timestamp` | Int64 | Момент события в миллисекундах от UNIX-эпохи (naive MSK через `.dt.epoch("ms")` — то есть мс соответствует MSK-датавремени, представленному как если бы оно было UTC). |
| `eid` | UInt32 | Тип события: показ, клик, добавление в избранное, контакт и др. Анонимизирован. |
| `user_id` | UInt32 | Анонимизированный идентификатор пользователя (0-based plain int). |
| `item_id` | UInt32 | Анонимизированный идентификатор объявления. |

In [16]:
eval_users = (pl.scan_parquet('../data/eval_user_events.pq')
      .with_columns(pl.col('timestamp').cast(pl.Datetime('ms')))
      .collect()
)
eval_users

timestamp,eid,user_id,item_id
datetime[ms],u32,u32,u32
2026-01-21 08:50:40,7,33,145340854
2026-01-21 08:51:07,7,33,69844009
2026-01-21 08:51:40,7,33,5573214
2026-01-21 22:27:43,7,33,40031
2026-01-21 22:28:29,10,33,40031
…,…,…,…
2026-04-13 17:58:16,7,8427567,31924474
2026-04-14 15:54:31,9,8427684,61572740
2026-04-14 15:54:32,14,8427684,61572740


In [23]:
print(f"Диапазон дат: {eval_users['timestamp'].min()} - {eval_users['timestamp'].max()}")

print(f"Уникальных действий: {eval_users['eid'].n_unique()}")

eval_users['eid'].value_counts().sort('count')

Диапазон дат: 2026-01-21 00:00:00 - 2026-04-14 23:59:59
Уникальных действий: 17


eid,count
u32,u32
0,245
8,1702
5,8738
2,10635
16,104937
…,…
9,819748
1,1035910
4,1575728


In [25]:
eval_users.filter(pl.col('eid').is_in([0,2,4,6,8,9,11,14,15,16]))

timestamp,eid,user_id,item_id
datetime[ms],u32,u32,u32
2026-01-22 09:30:45,4,33,59507938
2026-01-22 09:33:40,4,33,40031
2026-01-26 12:57:02,4,33,105996923
2026-01-26 20:15:32,4,33,19549508
2026-02-11 17:00:11,4,33,117140001
…,…,…,…
2026-02-21 16:56:19,4,8427555,154681513
2026-03-16 15:48:56,11,8427555,126823646
2026-03-16 15:49:44,4,8427555,126823646


In [27]:
eval_users.filter(pl.col('eid').is_in([0,2,4,6,8,9,11,14,15,16]))['eid'].value_counts().sort('count')

eid,count
u32,u32
0,245
8,1702
2,10635
16,104937
6,144681
11,288045
14,337903
15,750665
9,819748


# тест

In [30]:
test_users = eval_users.filter(pl.col('user_id').is_in([33]))
test_user = test_users.join(items, on='item_id', how='left')
test_user

timestamp,eid,user_id,item_id,vertical_id,category_ext_y,region_id_y,loc_id_y,sid_0_y,sid_1_y,sid_2_y,sid_3_y
datetime[ms],u32,u32,u32,u8,u16,u16,u16,u16,u16,u16,u16
2026-01-21 08:50:40,7,33,145340854,3,28,53,2368,434,44,979,703
2026-01-21 08:51:07,7,33,69844009,3,28,53,2368,434,44,500,15
2026-01-21 08:51:40,7,33,5573214,5,0,53,2368,917,478,884,141
2026-01-21 22:27:43,7,33,40031,3,28,53,2368,434,493,235,697
2026-01-21 22:28:29,10,33,40031,3,28,53,2368,434,493,235,697
…,…,…,…,…,…,…,…,…,…,…,…
2026-04-14 21:45:41,7,33,73146872,0,4,53,2368,293,425,942,457
2026-04-14 21:47:54,10,33,73146872,0,4,53,2368,293,425,942,457
2026-04-14 21:48:04,7,33,89687997,5,3,53,2368,110,130,209,56
